##### 출력 파서(Output Parser)
----
LLM의 출력값을 구조화된 형식으로 변환하고 답변에서 우리가 원하는 정보만 뽑아낼 때 유용하게 사용되는 도구입니다.
출력을 **구조화**된 정보로 변환. **명확**하고 **체계적**인 형태로 정보 제공의 이점 
**특징**
- 다양성
- 스트리밍 지원
- 확장성 
**이점**
- 구조화 : 체계적 관리
- 일관성 : 후속 처리 및 조회가 쉽고 효율적
- 유연성 : 변환 기능 제공
###### 스키마
메일에서는 보낸 사람, 이메일 주소, 날짜, 형식 처럼 사전에 **정의된 양식**을 **스키마**라고 한다.
----
##### Pydantic
**유효성 검사** :데이터 조건과 형식을 검토, 오류를 사전 방지

##### PydanticOutputParser



In [3]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("03-OutputParser")

llm = ChatOpenAI(model="gpt-4o-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
03-OutputParser


In [7]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [8]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)
chain = prompt | llm 
answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

다음은 이메일의 중요한 내용입니다:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (Teddy International)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 요청 내용: "ZENESIS" 모델에 대한 상세한 브로슈어 (기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 meeting 
- 목적: 협력 가능성 논의 및 유통 전략, 마케팅 계획 구체화

In [9]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [10]:
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [11]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [13]:
prompt = PromptTemplate.from_template(
    """
    You are a helpful assistant. Please answer the following questions in KOREAN.
    
    QUESTION:
    {question}
    
    EMAIL CONVERSATION:
    {email_conversation}
    
    FORMAT:
    {format}
    """
)

In [15]:
prompt = prompt.partial(format=parser.get_format_instructions())
prompt


PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [16]:
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "ZENESIS 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "김철수 상무가 이은채 대리에게 바이크코퍼레이션의 ZENESIS 자전거 유통 협력을 제안하며, 브로슈어와 기술 사양 정보를 요청하고, 다음 주 화요일 오전 10시에 미팅을 제안함.",
  "date": "1월 15일 오전 10시"
}
```

In [18]:
structured_output = parser.parse(output)
structured_output

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='ZENESIS 자전거 유통 협력 및 미팅 일정 제안', summary='김철수 상무가 이은채 대리에게 바이크코퍼레이션의 ZENESIS 자전거 유통 협력을 제안하며, 브로슈어와 기술 사양 정보를 요청하고, 다음 주 화요일 오전 10시에 미팅을 제안함.', date='1월 15일 오전 10시')

In [28]:
'''class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")'''
structured_output.person

'김철수'

In [29]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출헤 주세요.",
    }
)
response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무가 이은채 대리님에게 ZENESIS 자전거 브로슈어 요청 및 1월 15일 화요일 오전 10시에 미팅 제안.', date='2024-01-15T10:00:00')

##### with_structured_output() 바인딩

In [31]:
llm_with_structured = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EmailSummary)

In [30]:
answer = llm_with_structured.invoke(email_conversation)
answer.person

'김철수'